# Notebook 01 — EDA, Feature Engineering & Preprocessing


## 0. Environment Setup


In [3]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR       = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

ModuleNotFoundError: No module named 'matplotlib'

---


In [ ]:
SELECTED_FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean',
    'Flow Bytes/s', 'Flow Packets/s', 'Init_Win_bytes_forward',
    'Active Mean', 'Idle Mean', 'Protocol', 'TCP Flags', 'Label',
]
COLUMN_ALIASES = {
    'Total Backward Packets': 'Total Bwd Packets',
    ' Label': 'Label', 'label': 'Label', ' Protocol': 'Protocol',
    'Fwd PSH Flags': 'TCP Flags',
}

def load_raw_csvs(raw_dir):
    csv_files = list(raw_dir.glob('*.csv'))
    if not csv_files:
        raise FileNotFoundError(
            f'No CSV files in {raw_dir}.\n'
            'Download CIC-DDoS2019 and place CSVs there, or run scripts/download_datasets.sh'
        )
    dfs = []
    for f in csv_files:
        print(f'  Loading {f.name} ...')
        df = pd.read_csv(f, low_memory=False)
        df.columns = df.columns.str.strip()
        df = df.rename(columns=COLUMN_ALIASES)
        dfs.append(df)
    merged = pd.concat(dfs, ignore_index=True)
    print(f'Merged shape: {merged.shape}')
    return merged

df_raw = load_raw_csvs(RAW_DIR)

for col in SELECTED_FEATURES:
    if col not in df_raw.columns:
        print(f'[WARN] Missing column, zero-filling: {col}')
        df_raw[col] = 0

df = df_raw[SELECTED_FEATURES].copy()
print(f'Total flow records: {len(df):,}')
assert len(df) > 0, 'Dataset is empty!'
print('[M6] ✅ Raw CSV loaded and schema validated.')

---


In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Descriptive Statistics ===')
display(df.describe().T.round(3))

In [ ]:
label_counts = df['Label'].value_counts()
print(f'Label distribution:\n{label_counts}')
fig, ax = plt.subplots(figsize=(8, 4))
label_counts.plot(kind='bar', color=sns.color_palette('Set2', len(label_counts)),
                  edgecolor='black', ax=ax)
ax.set_title('Class Distribution (Raw)', fontsize=14, fontweight='bold')
ax.set_xlabel('Label'); ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_class_distribution.png', bbox_inches='tight')
plt.show()
print(f'Imbalance ratio: {label_counts.max() / label_counts.min():.1f}:1')

In [ ]:
df_numeric = df.select_dtypes(include=[np.number])
n_inf = np.isinf(df_numeric.values).sum()
n_nan = df.isna().sum().sum()
print(f'Infinite values: {n_inf:,}  |  NaN values: {n_nan:,}')
nan_per_col = df.isna().sum()
if nan_per_col.any():
    print(nan_per_col[nan_per_col > 0])

In [ ]:
CONT_COLS = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Fwd Packet Length Max', 'Flow Bytes/s', 'Flow Packets/s',
    'Active Mean', 'Idle Mean',
]
avail_cont = [c for c in CONT_COLS if c in df.columns]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, col in enumerate(avail_cont[:8]):
    data = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    axes.flatten()[i].hist(data.clip(upper=data.quantile(0.99)), bins=60,
                           color='steelblue', edgecolor='none', alpha=0.85)
    axes.flatten()[i].set_title(col, fontsize=9)
plt.suptitle('Feature Distributions (raw, 99th pct clipped)', fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_feature_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
df_corr = df.copy()
if df_corr['Label'].dtype == object:
    df_corr['Label'] = df_corr['Label'].str.upper().ne('BENIGN').astype(int)
num_corr_cols = df_corr.select_dtypes(include=[np.number]).columns
corr = df_corr[num_corr_cols].replace([np.inf, -np.inf], np.nan).corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, linewidths=0.3,
            vmin=-1, vmax=1, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_correlation_matrix.png', bbox_inches='tight')
plt.show()
if 'Label' in corr:
    top = corr['Label'].drop('Label').abs().sort_values(ascending=False).head(10)
    print('Top 10 features correlated with Label:'); print(top.to_string())
print('[M7] ✅ EDA complete.')

---


In [ ]:
from src.fl_client.dataset import (
    encode_categoricals, fit_scaler, save_scaler, build_dataloaders, load_scaler,
    CONTINUOUS_FEATURES, CATEGORICAL_FEATURES, TARGET_COLUMN
)

df_clean = df.copy().replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.rename(columns={'Total Backward Packets': 'Total Bwd Packets'})
num_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

df_encoded = encode_categoricals(df_clean)
if df_encoded[TARGET_COLUMN].dtype == object:
    df_encoded[TARGET_COLUMN] = df_encoded[TARGET_COLUMN].str.upper().ne('BENIGN').astype(int)

for col in CONTINUOUS_FEATURES:
    if col not in df_encoded.columns:
        df_encoded[col] = 0.0

scaler = fit_scaler(df_encoded)
df_encoded[CONTINUOUS_FEATURES] = scaler.transform(
    df_encoded[CONTINUOUS_FEATURES].values.astype(np.float32)
)

means = df_encoded[CONTINUOUS_FEATURES].mean()
print('Post-normalization mean (should be ≈ 0):')
print(means.round(3))
print('[M8] ✅ Normalization verified.')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, col in enumerate(CONTINUOUS_FEATURES[:8]):
    axes.flatten()[i].hist(df_encoded[col].clip(-4, 4), bins=60,
                           color='darkorange', edgecolor='none', alpha=0.85)
    axes.flatten()[i].set_title(f'{col} (normalised)', fontsize=9)
plt.suptitle('After QuantileTransformer (clipped ±4σ)', fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_normalized_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
out_csv = PROCESSED_DIR / 'cicddos2019_processed.csv'
output_cols = CONTINUOUS_FEATURES + CATEGORICAL_FEATURES + [TARGET_COLUMN]
df_out = df_encoded[[c for c in output_cols if c in df_encoded.columns]]
df_out.to_csv(out_csv, index=False)
print(f'Processed CSV: {out_csv}  ({len(df_out):,} rows)')

save_scaler(scaler, PROCESSED_DIR)
print(f'Scaler saved: {PROCESSED_DIR / "quantile_scaler.pkl"}')

frozen_scaler = load_scaler(PROCESSED_DIR / 'quantile_scaler.pkl')
train_loader, val_loader, _ = build_dataloaders(out_csv, scaler=frozen_scaler, batch_size=256)
x_cont, x_cat, y = next(iter(train_loader))
print(f'x_cont: {x_cont.shape}  x_cat: {x_cat.shape}  y: {y.shape}')
assert x_cont.shape[1] == len(CONTINUOUS_FEATURES)
print('[M8] ✅ DataLoader tensors validated. Phase 2 data pipeline complete.')

## Summary
